# Chavruta.AI — Embed הלכה / Halakhah (Kaggle GPU, per-part)

Embeds **one part** of the full Sefaria Halakhah library (רמב"ם · טור · שו"ע + נושאי כלים · ספרי מצוות · ראשונים · אחרונים · מודרני) with **bge-m3** — same format/collection as Tanakh / Mishnah / Gemara / שו"ת.

The library is sharded by `scripts/fetch_halacha.py --next K` into `halacha_partNN.jsonl`. Set `PART` below, run, download, load — then bump `PART` and repeat until done.

Settings: Accelerator = **GPU T4 x2 / P100** · Internet = **On**.

> Alternative to this notebook: run the **Nebius Job** (`scripts/ingest_job.py`) with `INGEST_CORPUS_HF_FILE=halacha_partNN.json` + `INGEST_INDEX_HF_REPO=...` — it embeds on a Nebius GPU and publishes the index straight to HF.

In [ ]:
PART = 1  # ← which part to embed (matches halacha_partNN.jsonl)

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q -U FlagEmbedding huggingface_hub numpy

In [ ]:
# Pull this part from the HF dataset. Upload it once from your machine first:
#   python scripts/upload_dataset_hf.py --repo Yehuda-Rubin/chavruta-torah-mixed \
#       --file data/processed/halacha/halacha_part01.jsonl --name halacha_part01.jsonl
import json
from pathlib import Path
from huggingface_hub import hf_hub_download

HF_DATASET = 'Yehuda-Rubin/chavruta-torah-mixed'
fname = f'halacha_part{PART:02d}.jsonl'
p = hf_hub_download(repo_id=HF_DATASET, filename=fname,
                    repo_type='dataset', local_dir='/kaggle/working')
print('downloaded:', p)

In [ ]:
jsonl_path = Path(f'/kaggle/working/halacha_part{PART:02d}.jsonl')
chunks = [json.loads(l) for l in jsonl_path.open(encoding='utf-8')]
docs  = [c['document'] for c in chunks]
ids   = [c['id']       for c in chunks]
metas = [c['metadata'] for c in chunks]
print(f'part {PART:02d}: {len(chunks):,} chunks')
from collections import Counter
print('by period:', Counter(m.get('period') for m in metas))

In [ ]:
from FlagEmbedding import BGEM3FlagModel
model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True, device='cuda')
print('model ready')

In [ ]:
import numpy as np, time, pickle

BATCH, MAX_LEN, CKPT_EVERY = 128, 512, 10000
ckpt = Path(f'/kaggle/working/embed_halacha_part{PART:02d}_ckpt.pkl')

if ckpt.exists():
    state = pickle.loads(ckpt.read_bytes())
    dense_parts, sparse_rows, start = state['dense'], state['sparse'], state['done']
    print(f'resuming from {start:,}')
else:
    dense_parts, sparse_rows, start = [], [], 0

t0 = time.time()
for s in range(start, len(docs), BATCH):
    batch = docs[s:s+BATCH]
    enc = model.encode(batch, batch_size=BATCH, max_length=MAX_LEN,
                       return_dense=True, return_sparse=True, return_colbert_vecs=False)
    dense_parts.append(np.asarray(enc['dense_vecs'], dtype='float32'))
    for w in enc['lexical_weights']:
        sparse_rows.append({int(t): float(v) for t, v in dict(w).items()})
    done = min(s + BATCH, len(docs))
    if done % CKPT_EVERY < BATCH or done == len(docs):
        ckpt.write_bytes(pickle.dumps({'dense': dense_parts, 'sparse': sparse_rows, 'done': done}))
    rate = done / max(time.time() - t0, 1e-9)
    eta = (len(docs) - done) / max(rate, 1e-9) / 60
    print(f'  {done:,}/{len(docs):,}  ({rate:.0f}/s, ETA {eta:.0f} min)', end='\r')

vecs = np.vstack(dense_parts)
norms = np.linalg.norm(vecs, axis=1, keepdims=True); norms[norms==0] = 1.0
vecs /= norms
print(f'\ndone: {vecs.shape} in {(time.time()-t0)/60:.1f} min')

In [ ]:
out = Path(f'/kaggle/working/out_halacha_part{PART:02d}'); out.mkdir(exist_ok=True)

np.save(out / 'corpus_vectors.npy', vecs)
del vecs

with open(out / 'corpus_sparse.jsonl', 'w', encoding='utf-8') as f:
    for i, row in enumerate(sparse_rows):
        f.write(json.dumps({'i': i, 'sparse': row}) + '\n')
del sparse_rows

with open(out / 'corpus_meta.jsonl', 'w', encoding='utf-8') as f:
    for i, (cid, doc, meta) in enumerate(zip(ids, docs, metas)):
        f.write(json.dumps({'i': i, 'id': cid, 'document': doc, 'metadata': meta},
                           ensure_ascii=False) + '\n')

for p in sorted(out.iterdir()):
    print(f'{p.name:28s} {p.stat().st_size/1e6:8.1f} MB')

In [ ]:
import shutil
archive = f'/kaggle/working/halacha_part{PART:02d}_vectors'
shutil.make_archive(archive, 'zip', out)
print(f'download: {archive}.zip  ({Path(archive + ".zip").stat().st_size/1e6:.0f} MB)')

## After download — add this part to the SAME RAG
```powershell
Expand-Archive halacha_part01_vectors.zip -DestinationPath out_halacha_part01
# --no-recreate ADDS to the existing collection (Tanakh + Mishnah + Gemara + שו"ת)
python scripts/load_to_store.py --in out_halacha_part01 --no-recreate --profile local
```
Then bump `PART` and repeat for every part until `fetch_halacha.py --plan` reports **ALL WORKS DONE**.
Finally (once), refresh Hebrew lexical search: `python scripts/backfill_search_he.py`.